In [1]:
"""
Mini Autograd Engine — a tiny reverse-mode autodiff library from scratch.
Inspired by Karpathy's micrograd.  No PyTorch needed for the engine itself.

Supports: +, -, *, /, **, relu, sigmoid, tanh, exp, log
"""

import math

# ──────────────────────────────────────────────
# 1. The Value node — heart of the engine
# ──────────────────────────────────────────────

class Value:
    """
    Stores a scalar value and its gradient.
    Builds a computation graph automatically via operator overloading.
    """

    def __init__(self, data: float, _children: tuple = (), _op: str = "", label: str = ""):
        self.data     = float(data)
        self.grad     = 0.0              # dL/d(self)
        self.label    = label
        self._op      = _op              # operation that produced this node
        self._prev    = set(_children)   # parent nodes
        self._backward = lambda: None    # local backward function

    # ── Pretty printing ──
    def __repr__(self):
        name = f"({self.label})" if self.label else ""
        return f"Value{name}(data={self.data:.4f}, grad={self.grad:.4f})"

    # ── Addition ──
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad  += out.grad       # d(a+b)/da = 1
            other.grad += out.grad       # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __radd__(self, other):  # other + self
        return self + other

    # ── Multiplication ──
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad  += other.data * out.grad    # d(a*b)/da = b
            other.grad += self.data  * out.grad    # d(a*b)/db = a
        out._backward = _backward
        return out

    def __rmul__(self, other):  # other * self
        return self * other

    # ── Negation & Subtraction ──
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return (-self) + other

    # ── Division ──
    def __truediv__(self, other):
        return self * (other ** -1)

    def __rtruediv__(self, other):  # other / self
        return other * (self ** -1)

    # ── Power ──
    def __pow__(self, exp):
        assert isinstance(exp, (int, float)), "only int/float exponents"
        out = Value(self.data ** exp, (self,), f"**{exp}")

        def _backward():
            self.grad += exp * (self.data ** (exp - 1)) * out.grad
        out._backward = _backward
        return out

    # ── Activations ──
    def relu(self):
        out = Value(max(0, self.data), (self,), "relu")

        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + math.exp(-self.data))
        out = Value(s, (self,), "σ")

        def _backward():
            self.grad += s * (1 - s) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")

        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    # ── Backward pass (reverse-mode autodiff) ──
    def backward(self):
        """
        Topological sort → reverse → call _backward() on each node.
        This propagates gradients from output back to all inputs.
        """
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0   # dL/dL = 1
        for node in reversed(topo):
            node._backward()


# ──────────────────────────────────────────────
# 2. Mini neural network layers
# ──────────────────────────────────────────────

import random

class Neuron:
    """Single neuron: w·x + b → activation."""

    def __init__(self, n_in: int, activation: str = "relu"):
        self.w = [Value(random.uniform(-1, 1), label=f"w{i}") for i in range(n_in)]
        self.b = Value(0.0, label="b")
        self.activation = activation

    def __call__(self, x: list) -> Value:
        # w·x + b
        out = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        if self.activation == "relu":
            return out.relu()
        elif self.activation == "sigmoid":
            return out.sigmoid()
        elif self.activation == "tanh":
            return out.tanh()
        return out  # "linear"

    def parameters(self) -> list:
        return self.w + [self.b]


class Layer:
    """A layer of neurons."""

    def __init__(self, n_in: int, n_out: int, activation: str = "relu"):
        self.neurons = [Neuron(n_in, activation) for _ in range(n_out)]

    def __call__(self, x: list) -> list:
        outs = [n(x) for n in self.neurons]
        return outs if len(outs) > 1 else outs[0]

    def parameters(self) -> list:
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    """Multi-layer perceptron."""

    def __init__(self, layer_sizes: list, activations: list = None):
        """
        layer_sizes: e.g. [2, 8, 4, 1]
        activations: e.g. ["relu", "relu", "sigmoid"]  (one per layer)
        """
        if activations is None:
            activations = ["relu"] * (len(layer_sizes) - 2) + ["sigmoid"]
        self.layers = [Layer(layer_sizes[i], layer_sizes[i+1], activations[i])
                       for i in range(len(layer_sizes) - 1)]

    def __call__(self, x: list):
        for layer in self.layers:
            x = layer(x) if isinstance(layer(x), list) else [layer(x)]
        return x[0] if len(x) == 1 else x

    def parameters(self) -> list:
        return [p for layer in self.layers for p in layer.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0


# ──────────────────────────────────────────────
# 3. Example usage
# ──────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 55)
    print("  Mini Autograd Engine")
    print("=" * 55)

    # ── Demo 1: Basic computation + backward ──
    print("\n── Demo 1: Basic forward + backward ──\n")

    a = Value(2.0, label="a")
    b = Value(-3.0, label="b")
    c = Value(10.0, label="c")

    # f = a*b + c
    d = a * b;     d.label = "d=a*b"
    f = d + c;     f.label = "f=d+c"

    print(f"  Forward:  a={a.data}, b={b.data}, c={c.data}")
    print(f"  f = a*b + c = {f.data}")

    f.backward()

    print(f"\n  Backward (df/d...):")
    print(f"    df/da = b = {a.grad:.4f}     (expected: {b.data})")
    print(f"    df/db = a = {b.grad:.4f}     (expected: {a.data})")
    print(f"    df/dc = 1 = {c.grad:.4f}     (expected: 1.0)")

    # ── Demo 2: More complex expression ──
    print("\n── Demo 2: Complex expression ──\n")

    x = Value(3.0, label="x")
    y = Value(4.0, label="y")

    # z = (x * y + x^2) / y
    z = (x * y + x ** 2) / y
    z.label = "z"

    print(f"  z = (x*y + x²) / y")
    print(f"  z = ({x.data}*{y.data} + {x.data}²) / {y.data} = {z.data:.4f}")

    z.backward()
    print(f"  dz/dx = 1 + 2x/y = {x.grad:.4f}  (expected: {1 + 2*3/4:.4f})")
    print(f"  dz/dy = -x²/y²   = {y.grad:.4f}  (expected: {-9/16:.4f})")

    # ── Demo 3: Activation functions ──
    print("\n── Demo 3: Activations ──\n")

    for name, fn in [("relu", "relu"), ("sigmoid", "sigmoid"), ("tanh", "tanh")]:
        v = Value(0.5, label="v")
        out = getattr(v, fn)()
        out.backward()
        print(f"  {name:>7}(0.5) = {out.data:.4f}   grad = {v.grad:.4f}")

    # ── Demo 4: Verify against PyTorch ──
    print("\n── Demo 4: Verify against PyTorch ──\n")

    try:
        import torch

        # Same computation in both engines
        # f = sigmoid(a*b + c^2)
        a_v = Value(1.5, label="a")
        b_v = Value(-2.0, label="b")
        c_v = Value(0.5, label="c")
        f_v = (a_v * b_v + c_v ** 2).sigmoid()
        f_v.backward()

        a_t = torch.tensor(1.5, requires_grad=True)
        b_t = torch.tensor(-2.0, requires_grad=True)
        c_t = torch.tensor(0.5, requires_grad=True)
        f_t = torch.sigmoid(a_t * b_t + c_t ** 2)
        f_t.backward()

        print(f"  Expression: sigmoid(a*b + c²)")
        print(f"  {'':>12} {'Ours':>10} {'PyTorch':>10} {'Match':>8}")
        print(f"  {'f':>12} {f_v.data:>10.6f} {f_t.item():>10.6f} {'✓' if abs(f_v.data - f_t.item()) < 1e-6 else '✗':>8}")
        print(f"  {'df/da':>12} {a_v.grad:>10.6f} {a_t.grad.item():>10.6f} {'✓' if abs(a_v.grad - a_t.grad.item()) < 1e-6 else '✗':>8}")
        print(f"  {'df/db':>12} {b_v.grad:>10.6f} {b_t.grad.item():>10.6f} {'✓' if abs(b_v.grad - b_t.grad.item()) < 1e-6 else '✗':>8}")
        print(f"  {'df/dc':>12} {c_v.grad:>10.6f} {c_t.grad.item():>10.6f} {'✓' if abs(c_v.grad - c_t.grad.item()) < 1e-6 else '✗':>8}")
    except ImportError:
        print("  (PyTorch not installed — skipping verification)")

    # ── Demo 5: Train a tiny neural network on XOR ──
    print("\n── Demo 5: Train MLP on XOR ──\n")

    random.seed(42)

    # XOR dataset
    X = [[0, 0], [0, 1], [1, 0], [1, 1]]
    Y = [0, 1, 1, 0]

    # 2 → 8 → 1 network
    net = MLP([2, 8, 1], activations=["tanh", "sigmoid"])
    print(f"  Parameters: {len(net.parameters())}")
    print(f"  Architecture: 2 → 8 (tanh) → 1 (sigmoid)\n")

    LR = 0.5
    for epoch in range(1, 201):
        # Forward: compute predictions and MSE loss
        preds = [net(x) for x in X]
        loss = sum((p - y) ** 2 for p, y in zip(preds, Y))

        # Backward
        net.zero_grad()
        loss.backward()

        # Update (SGD)
        for p in net.parameters():
            p.data -= LR * p.grad

        if epoch % 50 == 0:
            acc = sum(1 for p, y in zip(preds, Y)
                      if (p.data >= 0.5) == (y == 1)) / len(Y)
            print(f"  Epoch {epoch:3d} | Loss: {loss.data:.4f} | Acc: {acc:.0%}")

    # Final predictions
    print("\n  Final predictions:")
    for x, y in zip(X, Y):
        p = net(x)
        print(f"    {x} → {p.data:.4f}  (target: {y})")

    # ── How it works ──
    print(f"""
── How the engine works ──

  1. FORWARD:  Each operation (+, *, relu, ...) creates a new Value node
               and records which Values produced it (_prev) and how (_op).

  2. GRAPH:    This builds a directed acyclic graph (DAG) of the entire
               computation — just like PyTorch's autograd tape.

  3. BACKWARD: .backward() does a topological sort of the graph,
               then walks it in reverse, calling each node's local
               _backward() to propagate gradients via the chain rule.

  4. RESULT:   Every Value.grad now holds dL/d(value) — the gradient
               of the loss with respect to that value.
""")

  Mini Autograd Engine

── Demo 1: Basic forward + backward ──

  Forward:  a=2.0, b=-3.0, c=10.0
  f = a*b + c = 4.0

  Backward (df/d...):
    df/da = b = -3.0000     (expected: -3.0)
    df/db = a = 2.0000     (expected: 2.0)
    df/dc = 1 = 1.0000     (expected: 1.0)

── Demo 2: Complex expression ──

  z = (x*y + x²) / y
  z = (3.0*4.0 + 3.0²) / 4.0 = 5.2500
  dz/dx = 1 + 2x/y = 2.5000  (expected: 2.5000)
  dz/dy = -x²/y²   = -0.5625  (expected: -0.5625)

── Demo 3: Activations ──

     relu(0.5) = 0.5000   grad = 1.0000
  sigmoid(0.5) = 0.6225   grad = 0.2350
     tanh(0.5) = 0.4621   grad = 0.7864

── Demo 4: Verify against PyTorch ──

  Expression: sigmoid(a*b + c²)
                     Ours    PyTorch    Match
             f   0.060087   0.060087        ✓
         df/da  -0.112952  -0.112952        ✓
         df/db   0.084714   0.084714        ✓
         df/dc   0.056476   0.056476        ✓

── Demo 5: Train MLP on XOR ──

  Parameters: 33
  Architecture: 2 → 8 (tanh) → 1 (sig